In [ ]:
import random
import time
import numpy as np
import torch

from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 200kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.76MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 23.1MB/s]


In [ ]:
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=256,
    shuffle=False
)

In [ ]:
class FashionMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

        self.activation = nn.ReLU()

    def forward(self, x):

        x = self.flatten(x)

        x = self.activation(self.fc1(x))

        x = self.activation(self.fc2(x))

        logits = self.fc3(x)

        return logits

model = FashionMLP().to(device)

print(model)

FashionMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
  (activation): ReLU()
)


In [ ]:
print(model)

FashionMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
  (activation): ReLU()
)


In [ ]:
images, labels = next(iter(train_loader))

# Move data to the same device as the model
images = images.to(device)
labels = labels.to(device)

print("Input:", images.shape)

x = model.flatten(images)
print("After Flatten:", x.shape)

x = model.activation(model.fc1(x))
print("After Layer1:", x.shape)

x = model.activation(model.fc2(x))
print("After Layer2:", x.shape)

x = model.fc3(x)
print("After Layer3:", x.shape)

Input: torch.Size([64, 1, 28, 28])
After Flatten: torch.Size([64, 784])
After Layer1: torch.Size([64, 128])
After Layer2: torch.Size([64, 64])
After Layer3: torch.Size([64, 10])


In [ ]:
total_params = sum(p.numel() for p in model.parameters())

print("Total Parameters:", total_params)

Total Parameters: 109386


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        correct += (logits.argmax(1) == labels).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
def evaluate(model, loader):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)

            loss = criterion(logits, labels)

            running_loss += loss.item() * labels.size(0)

            correct += (logits.argmax(1) == labels).sum().item()

            total += labels.size(0)

    loss = running_loss / total
    accuracy = correct / total

    return loss, accuracy

In [ ]:
epochs = 10

train_losses = []
test_losses = []

train_accuracies = []
test_accuracies = []

for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(model, train_loader)

    test_loss, test_acc = evaluate(model, test_loader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(f"Epoch {epoch+1}/{epochs}")

    print(f"Train Loss : {train_loss:.4f}")

    print(f"Train Accuracy : {train_acc:.4f}")

    print(f"Test Loss : {test_loss:.4f}")

    print(f"Test Accuracy : {test_acc:.4f}")

    print("-"*40)

Epoch 1/10
Train Loss : 0.5144
Train Accuracy : 0.8141
Test Loss : 0.4465
Test Accuracy : 0.8374
----------------------------------------
Epoch 2/10
Train Loss : 0.3811
Train Accuracy : 0.8614
Test Loss : 0.3993
Test Accuracy : 0.8573
----------------------------------------
Epoch 3/10
Train Loss : 0.3444
Train Accuracy : 0.8727
Test Loss : 0.3599
Test Accuracy : 0.8714
----------------------------------------
Epoch 4/10
Train Loss : 0.3166
Train Accuracy : 0.8832
Test Loss : 0.3598
Test Accuracy : 0.8721
----------------------------------------
Epoch 5/10
Train Loss : 0.2986
Train Accuracy : 0.8898
Test Loss : 0.3510
Test Accuracy : 0.8764
----------------------------------------
Epoch 6/10
Train Loss : 0.2852
Train Accuracy : 0.8940
Test Loss : 0.3416
Test Accuracy : 0.8721
----------------------------------------
Epoch 7/10
Train Loss : 0.2694
Train Accuracy : 0.9001
Test Loss : 0.3474
Test Accuracy : 0.8756
----------------------------------------
Epoch 8/10
Train Loss : 0.2583
Tra

In [34]:
from sklearn.metrics import f1_score

In [ ]:
class FashionMLP(nn.Module):

    def __init__(self, hidden1=128, activation="relu"):
        super().__init__()

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(784, hidden1)
        self.fc2 = nn.Linear(hidden1, 64)
        self.fc3 = nn.Linear(64, 10)

        if activation.lower() == "relu":
            self.activation = nn.ReLU()
        elif activation.lower() == "tanh":
            self.activation = nn.Tanh()
        else:
            raise ValueError("Unsupported activation")

    def forward(self, x):
        x = self.flatten(x)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)

        return x

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [35]:
def run_experiment(hidden1, activation, optimizer_name, lr, epochs=10):

    model = FashionMLP(hidden1, activation).to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    start_train = time.time()

    # ---------- TRAIN ----------
    for epoch in range(epochs):

        model.train()

        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * labels.size(0)

            correct += (outputs.argmax(1) == labels).sum().item()

            total += labels.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total

    train_time = time.time() - start_train

    # ---------- TEST ----------
    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    start_test = time.time()

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0)

            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()

            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    inference_time = time.time() - start_test

    test_loss = running_loss / total
    test_accuracy = correct / total

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return {
        "Parameters": count_parameters(model),
        "Train Loss": train_loss,
        "Test Loss": test_loss,
        "Test Accuracy": test_accuracy,
        "Macro F1": macro_f1,
        "Train Time": train_time,
        "Inference Time": inference_time
    }

In [36]:
resultA = run_experiment(128, "relu", "SGD", 0.01)
print(resultA)

resultB = run_experiment(128, "relu", "Adam", 0.001)
print(resultB)

resultC = run_experiment(256, "relu", "Adam", 0.001)
print(resultC)

resultD = run_experiment(128, "tanh", "Adam", 0.001)
print(resultD)

{'Parameters': 109386, 'Train Loss': 0.3700658103942871, 'Test Loss': 0.4059122618675232, 'Test Accuracy': 0.8544, 'Macro F1': 0.8536026945728528, 'Train Time': 139.86491298675537, 'Inference Time': 1.8435273170471191}
{'Parameters': 109386, 'Train Loss': 0.23641692759195965, 'Test Loss': 0.35241964229345324, 'Test Accuracy': 0.8816, 'Macro F1': 0.8797175272283857, 'Train Time': 140.81111669540405, 'Inference Time': 2.3457133769989014}
{'Parameters': 218058, 'Train Loss': 0.22588450440565744, 'Test Loss': 0.36773720920085906, 'Test Accuracy': 0.8765, 'Macro F1': 0.8769566219174901, 'Train Time': 139.23785948753357, 'Inference Time': 1.957932472229004}
{'Parameters': 109386, 'Train Loss': 0.25961760083039603, 'Test Loss': 0.3445552343845367, 'Test Accuracy': 0.8765, 'Macro F1': 0.8763733766185151, 'Train Time': 137.6976273059845, 'Inference Time': 1.9451584815979004}
